In [ ]:
import os
import re
import json
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
ORIGINAL_CORPUS_DIR = Path("/content/original_corpus")
OUTPUT_DIR = Path("/content/reorganized_pair_split")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Original corpus directory:", ORIGINAL_CORPUS_DIR)
print("Output directory:", OUTPUT_DIR)

Original corpus directory: /content/original_corpus
Output directory: /content/reorganized_pair_split


In [ ]:
if not ORIGINAL_CORPUS_DIR.exists():
    raise FileNotFoundError(
        f"The folder {ORIGINAL_CORPUS_DIR} was not found. "
        "Create this folder in Colab and upload the original train, validation, and test JSONL files into it."
    )

jsonl_files = sorted(ORIGINAL_CORPUS_DIR.glob("*.jsonl"))

print("JSONL files found:")
for file in jsonl_files:
    print("-", file.name)

if len(jsonl_files) == 0:
    raise FileNotFoundError(f"No JSONL files were found in {ORIGINAL_CORPUS_DIR}.")

JSONL files found:
- test.jsonl
- train.jsonl
- validation.jsonl


In [ ]:
def find_jsonl_file(folder, split_name):
    folder = Path(folder)

    exact_names = {
        "train": ["train.jsonl"],
        "validation": ["validation.jsonl", "valid.jsonl", "val.jsonl"],
        "test": ["test.jsonl"]
    }

    for exact_name in exact_names[split_name]:
        exact_path = folder / exact_name
        if exact_path.exists():
            return exact_path

    candidates = sorted(folder.glob("*.jsonl"))

    if split_name == "train":
        patterns = ["train"]
    elif split_name == "validation":
        patterns = ["validation", "valid", "val"]
    elif split_name == "test":
        patterns = ["test"]
    else:
        raise ValueError(f"Unknown split name: {split_name}")

    matched_files = [
        file for file in candidates
        if any(pattern in file.name.lower() for pattern in patterns)
    ]

    if len(matched_files) == 0:
        raise FileNotFoundError(
            f"No JSONL file was found for split '{split_name}' in {folder}."
        )

    if len(matched_files) > 1:
        print(f"Warning: more than one file was found for split '{split_name}'. Using the first one:")
        for file in matched_files:
            print("-", file.name)

    return matched_files[0]

In [ ]:
train_path = find_jsonl_file(ORIGINAL_CORPUS_DIR, "train")
validation_path = find_jsonl_file(ORIGINAL_CORPUS_DIR, "validation")
test_path = find_jsonl_file(ORIGINAL_CORPUS_DIR, "test")

print("Files selected:")
print("Train:", train_path)
print("Validation:", validation_path)
print("Test:", test_path)

Files selected:
Train: /content/original_corpus/train.jsonl
Validation: /content/original_corpus/validation.jsonl
Test: /content/original_corpus/test.jsonl


In [ ]:
def load_jsonl(path, split_name):
    rows = []

    with open(path, "r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                item = json.loads(line)
                item["original_split"] = split_name
                rows.append(item)

    return rows

In [ ]:
all_rows = []
all_rows.extend(load_jsonl(train_path, "train"))
all_rows.extend(load_jsonl(validation_path, "validation"))
all_rows.extend(load_jsonl(test_path, "test"))

df = pd.DataFrame(all_rows)

print("Total examples:", len(df))
print("Columns:", list(df.columns))

display(df.head())

Total examples: 5700
Columns: ['id', 'text', 'label', 'tokens', 'labels', 'original_split']


,id,text,label,tokens,labels,original_split
0,5.46.H,Por que o carteiro foi à feira? Porque tinha u...,1,"[Por, que, o, carteiro, foi, à, feira, ?, Porq...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0]",train
1,5.792.H,Por que a mulher esotérica não conseguia engra...,1,"[Por, que, a, mulher, esotérica, não, consegui...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]",train
2,5.1811.H,Qual é o animal que está sempre cansado? Dorme...,1,"[Qual, é, o, animal, que, está, sempre, cansad...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1]",train
3,5.733.H,Qual o sambista passou a dar presente pra todo...,1,"[Qual, o, sambista, passou, a, dar, presente, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",train
4,4.652.N,Um homem matou uma ovelha e agora foi preso . ...,0,"[Um, homem, matou, uma, ovelha, e, agora, foi,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",train


In [ ]:
def load_jsonl(path, split_name):
    rows = []

    with open(path, "r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                item = json.loads(line)
                item["original_split"] = split_name
                rows.append(item)

    return rows

In [ ]:
all_rows = []
all_rows.extend(load_jsonl(train_path, "train"))
all_rows.extend(load_jsonl(validation_path, "validation"))
all_rows.extend(load_jsonl(test_path, "test"))

df = pd.DataFrame(all_rows)

print("Total examples:", len(df))
print("Columns:", list(df.columns))

display(df.head())

Total examples: 5700
Columns: ['id', 'text', 'label', 'tokens', 'labels', 'original_split']


,id,text,label,tokens,labels,original_split
0,5.46.H,Por que o carteiro foi à feira? Porque tinha u...,1,"[Por, que, o, carteiro, foi, à, feira, ?, Porq...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0]",train
1,5.792.H,Por que a mulher esotérica não conseguia engra...,1,"[Por, que, a, mulher, esotérica, não, consegui...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]",train
2,5.1811.H,Qual é o animal que está sempre cansado? Dorme...,1,"[Qual, é, o, animal, que, está, sempre, cansad...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1]",train
3,5.733.H,Qual o sambista passou a dar presente pra todo...,1,"[Qual, o, sambista, passou, a, dar, presente, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",train
4,4.652.N,Um homem matou uma ovelha e agora foi preso . ...,0,"[Um, homem, matou, uma, ovelha, e, agora, foi,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",train


In [ ]:
required_columns = {"id", "text", "label"}

missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"The following required columns are missing: {missing_columns}")

print("All required columns are present.")

All required columns are present.


In [ ]:
def extract_pair_id(example_id):
    return re.sub(r"\.[HN]$", "", str(example_id))

In [ ]:
def extract_pair_suffix(example_id):
    match = re.search(r"\.([HN])$", str(example_id))

    if match:
        return match.group(1)

    return None

In [ ]:
df["pair_id"] = df["id"].apply(extract_pair_id)
df["pair_suffix"] = df["id"].apply(extract_pair_suffix)

display(df[["id", "pair_id", "pair_suffix", "label", "original_split", "text"]].head())

,id,pair_id,pair_suffix,label,original_split,text
0,5.46.H,5.46,H,1,train,Por que o carteiro foi à feira? Porque tinha u...
1,5.792.H,5.792,H,1,train,Por que a mulher esotérica não conseguia engra...
2,5.1811.H,5.1811,H,1,train,Qual é o animal que está sempre cansado? Dorme...
3,5.733.H,5.733,H,1,train,Qual o sambista passou a dar presente pra todo...
4,4.652.N,4.652,N,0,train,Um homem matou uma ovelha e agora foi preso . ...


In [ ]:
print("Total examples:", len(df))
print("Total pairs:", df["pair_id"].nunique())

print("\nOriginal split distribution:")
display(df["original_split"].value_counts())

print("\nOriginal split and label distribution:")
display(pd.crosstab(df["original_split"], df["label"]))

print("\nPair suffix and label distribution:")
display(pd.crosstab(df["pair_suffix"], df["label"]))

pair_sizes = df.groupby("pair_id").size()

print("\nPair size distribution:")
display(pair_sizes.value_counts())

incomplete_pairs = pair_sizes[pair_sizes != 2]

if len(incomplete_pairs) > 0:
    print("Warning: some pairs do not have exactly two examples.")
    display(incomplete_pairs)
else:
    print("All pairs have exactly two examples.")

Total examples: 5700
Total pairs: 2850

Original split distribution:


,count
original_split,
train,3990
test,1140
validation,570



Original split and label distribution:


label,0,1
original_split,,
test,570,570
train,1995,1995
validation,285,285



Pair suffix and label distribution:


label,0,1
pair_suffix,,
H,0,2850
N,2850,0



Pair size distribution:


,count
2,2850


All pairs have exactly two examples.


In [ ]:
original_pair_split_counts = df.groupby("pair_id")["original_split"].nunique()
leaked_pairs_original = original_pair_split_counts[original_pair_split_counts > 1]

print("Number of pairs crossing different original splits:", len(leaked_pairs_original))

if len(leaked_pairs_original) > 0:
    print("Examples of leaked pairs in the original split:")

    sample_leaked_pair_ids = leaked_pairs_original.index[:10]

    display(
        df[df["pair_id"].isin(sample_leaked_pair_ids)]
        [["id", "pair_id", "pair_suffix", "label", "original_split", "text"]]
        .sort_values(["pair_id", "id"])
    )
else:
    print("No pair leakage was found in the original split.")

Number of pairs crossing different original splits: 1306
Examples of leaked pairs in the original split:


,id,pair_id,pair_suffix,label,original_split,text
4467,1.3.H,1.3,H,1,validation,"Se um pato perde a pata, ele fica manco ou viúvo?"
2492,1.3.N,1.3,N,0,train,"Se um pato perde a esposa, ele fica manco ou v..."
5069,1.4.H,1.4,H,1,test,O que uma impressora falou para a outra? Essa ...
914,1.4.N,1.4,N,0,train,O que uma impressora falou para a outra? Essa ...
1541,1.5.H,1.5,H,1,train,Por que a galinha bateu a cabeça contra a pare...
4346,1.5.N,1.5,N,0,validation,Por que a galinha bateu a cabeça contra a pare...
2669,2.12.H,2.12,H,1,train,Por que a vaca foi para o espaço? Para se enco...
4037,2.12.N,2.12,N,0,validation,Por que a vaca foi para o espaço? Para se enco...
220,2.14.H,2.14,H,1,train,Por que o astronauta cometeu um homicídio no e...
4604,2.14.N,2.14,N,0,test,Por que o astronauta cometeu um homicídio no e...


In [ ]:
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

original_example_counts = df["original_split"].value_counts().to_dict()

train_example_count = original_example_counts.get("train", 0)
validation_example_count = original_example_counts.get("validation", 0)
test_example_count = original_example_counts.get("test", 0)

for split_name, count in original_example_counts.items():
    if count % 2 != 0:
        raise ValueError(
            f"The original split '{split_name}' has {count} examples, "
            "which is not divisible by 2. Exact pair-level splitting is not possible."
        )

train_pair_count = train_example_count // 2
validation_pair_count = validation_example_count // 2
test_pair_count = test_example_count // 2

all_pair_ids = sorted(df["pair_id"].unique())
total_pairs = len(all_pair_ids)

expected_total_pairs = train_pair_count + validation_pair_count + test_pair_count

if expected_total_pairs != total_pairs:
    raise ValueError(
        f"Expected {expected_total_pairs} pairs based on original split sizes, "
        f"but found {total_pairs} unique pairs."
    )

In [ ]:
print("Target pair counts:")
print("Train pairs:", train_pair_count)
print("Validation pairs:", validation_pair_count)
print("Test pairs:", test_pair_count)
print("Total pairs:", expected_total_pairs)

shuffled_pair_ids = all_pair_ids.copy()
random.shuffle(shuffled_pair_ids)

train_pair_ids = set(shuffled_pair_ids[:train_pair_count])

validation_start = train_pair_count
validation_end = train_pair_count + validation_pair_count

validation_pair_ids = set(shuffled_pair_ids[validation_start:validation_end])

test_pair_ids = set(shuffled_pair_ids[validation_end:])

Target pair counts:
Train pairs: 1995
Validation pairs: 285
Test pairs: 570
Total pairs: 2850


In [ ]:
def assign_new_split(pair_id):
    if pair_id in train_pair_ids:
        return "train"

    if pair_id in validation_pair_ids:
        return "validation"

    if pair_id in test_pair_ids:
        return "test"

    raise ValueError(f"Pair ID not assigned to any split: {pair_id}")

In [ ]:
df["new_split"] = df["pair_id"].apply(assign_new_split)

print("\nNew split distribution:")
display(df["new_split"].value_counts())

print("\nNew split and label distribution:")
display(pd.crosstab(df["new_split"], df["label"]))


New split distribution:


,count
new_split,
train,3990
test,1140
validation,570



New split and label distribution:


label,0,1
new_split,,
test,570,570
train,1995,1995
validation,285,285


In [ ]:
new_pair_split_counts = df.groupby("pair_id")["new_split"].nunique()
leaked_pairs_new = new_pair_split_counts[new_pair_split_counts > 1]

print("Number of pairs crossing different new splits:", len(leaked_pairs_new))

if len(leaked_pairs_new) == 0:
    print("Success: no pair crosses train, validation, and test in the new split.")
else:
    print("Warning: pair leakage still exists in the new split.")
    display(leaked_pairs_new)

print("\nOriginal vs new split counts:")

split_comparison = pd.DataFrame({
    "original": df["original_split"].value_counts(),
    "new": df["new_split"].value_counts()
}).fillna(0).astype(int)

display(split_comparison)

print("\nNew label distribution by split:")
display(pd.crosstab(df["new_split"], df["label"]))

for split_name in ["train", "validation", "test"]:
    original_count = int((df["original_split"] == split_name).sum())
    new_count = int((df["new_split"] == split_name).sum())

    if original_count != new_count:
        raise ValueError(
            f"Split size mismatch for '{split_name}': "
            f"original={original_count}, new={new_count}"
        )

print("Success: the new split has exactly the same number of examples as the original split.")

Number of pairs crossing different new splits: 0
Success: no pair crosses train, validation, and test in the new split.

Original vs new split counts:


,original,new
train,3990,3990
test,1140,1140
validation,570,570



New label distribution by split:


label,0,1
new_split,,
test,570,570
train,1995,1995
validation,285,285


Success: the new split has exactly the same number of examples as the original split.


In [ ]:
def make_json_serializable(obj):
    if obj is None:
        return None

    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        if np.isnan(obj):
            return None
        return float(obj)

    if isinstance(obj, np.ndarray):
        return [make_json_serializable(value) for value in obj.tolist()]

    if isinstance(obj, (list, tuple)):
        return [make_json_serializable(value) for value in obj]

    if isinstance(obj, dict):
        return {
            str(key): make_json_serializable(value)
            for key, value in obj.items()
        }

    try:
        if pd.isna(obj):
            return None
    except (TypeError, ValueError):
        pass

    return obj

In [ ]:
def save_jsonl(dataframe, output_path):
    auxiliary_columns = {
        "original_split",
        "new_split",
        "pair_id",
        "pair_suffix"
    }

    columns_to_save = [
        column for column in dataframe.columns
        if column not in auxiliary_columns
    ]

    temp_output_path = output_path.with_suffix(output_path.suffix + ".tmp")

    with open(temp_output_path, "w", encoding="utf-8") as file:
        for _, row in dataframe[columns_to_save].iterrows():
            item = {
                key: make_json_serializable(value)
                for key, value in row.to_dict().items()
            }

            file.write(json.dumps(item, ensure_ascii=False) + "\n")

    temp_output_path.replace(output_path)

In [ ]:
train_new = df[df["new_split"] == "train"].copy()
validation_new = df[df["new_split"] == "validation"].copy()
test_new = df[df["new_split"] == "test"].copy()

save_jsonl(train_new, OUTPUT_DIR / "train.jsonl")
save_jsonl(validation_new, OUTPUT_DIR / "validation.jsonl")
save_jsonl(test_new, OUTPUT_DIR / "test.jsonl")

print("Reorganized corpus saved to:", OUTPUT_DIR)
print("-", OUTPUT_DIR / "train.jsonl")
print("-", OUTPUT_DIR / "validation.jsonl")
print("-", OUTPUT_DIR / "test.jsonl")

Reorganized corpus saved to: /content/reorganized_pair_split
- /content/reorganized_pair_split/train.jsonl
- /content/reorganized_pair_split/validation.jsonl
- /content/reorganized_pair_split/test.jsonl


In [ ]:
split_report = {
    "random_seed": RANDOM_SEED,
    "total_examples": int(len(df)),
    "total_pairs": int(df["pair_id"].nunique()),
    "original_split_counts": {
        split: int(count)
        for split, count in df["original_split"].value_counts().to_dict().items()
    },
    "new_split_counts": {
        split: int(count)
        for split, count in df["new_split"].value_counts().to_dict().items()
    },
    "original_label_distribution": {
        str(label): int(count)
        for label, count in df["label"].value_counts().to_dict().items()
    },
    "new_label_distribution_by_split": {
        split: {
            str(label): int(count)
            for label, count in group["label"].value_counts().to_dict().items()
        }
        for split, group in df.groupby("new_split")
    },
    "original_leaked_pairs": int(len(leaked_pairs_original)),
    "new_leaked_pairs": int(len(leaked_pairs_new)),
    "exact_same_split_sizes": bool(
        df["original_split"].value_counts().sort_index().equals(
            df["new_split"].value_counts().sort_index()
        )
    )
}

split_report_path = OUTPUT_DIR / "split_report.json"

with open(split_report_path, "w", encoding="utf-8") as file:
    json.dump(split_report, file, ensure_ascii=False, indent=4)

print("Split report saved to:", split_report_path)
print(json.dumps(split_report, ensure_ascii=False, indent=4))

Split report saved to: /content/reorganized_pair_split/split_report.json
{
    "random_seed": 42,
    "total_examples": 5700,
    "total_pairs": 2850,
    "original_split_counts": {
        "train": 3990,
        "test": 1140,
        "validation": 570
    },
    "new_split_counts": {
        "train": 3990,
        "test": 1140,
        "validation": 570
    },
    "original_label_distribution": {
        "1": 2850,
        "0": 2850
    },
    "new_label_distribution_by_split": {
        "test": {
            "0": 570,
            "1": 570
        },
        "train": {
            "1": 1995,
            "0": 1995
        },
        "validation": {
            "1": 285,
            "0": 285
        }
    },
    "original_leaked_pairs": 1306,
    "new_leaked_pairs": 0,
    "exact_same_split_sizes": true
}


In [ ]:
inspection_columns = [
    "id",
    "pair_id",
    "pair_suffix",
    "label",
    "original_split",
    "new_split",
    "text"
]

available_inspection_columns = [
    column for column in inspection_columns
    if column in df.columns
]

inspection_path = OUTPUT_DIR / "corpus_reorganized_inspection.csv"

df[available_inspection_columns].to_csv(
    inspection_path,
    index=False,
    encoding="utf-8"
)

print("Inspection CSV saved to:", inspection_path)

display(df[available_inspection_columns].head(20))

Inspection CSV saved to: /content/reorganized_pair_split/corpus_reorganized_inspection.csv


,id,pair_id,pair_suffix,label,original_split,new_split,text
0,5.46.H,5.46,H,1,train,validation,Por que o carteiro foi à feira? Porque tinha u...
1,5.792.H,5.792,H,1,train,train,Por que a mulher esotérica não conseguia engra...
2,5.1811.H,5.1811,H,1,train,validation,Qual é o animal que está sempre cansado? Dorme...
3,5.733.H,5.733,H,1,train,train,Qual o sambista passou a dar presente pra todo...
4,4.652.N,4.652,N,0,train,train,Um homem matou uma ovelha e agora foi preso . ...
5,5.2585.N,5.2585,N,0,train,train,Qual apresentador de TV vive gripado? Fausto S...
6,5.1990.N,5.1990,N,0,train,validation,Uma cerveja se associou a um comediante para a...
7,4.103.H,4.103,H,1,train,validation,Qual é a única coisa que se faz sempre em nume...
8,5.28.N,5.28,N,0,train,train,Qual é a modelo mais bela que existe? Gisele B...
9,4.66.N,4.66,N,0,train,test,Eu adoro fiambre . E também queijo.


In [ ]:
def count_jsonl_lines(path):
    with open(path, "r", encoding="utf-8") as file:
        return sum(1 for line in file if line.strip())

In [ ]:
saved_counts = {
    "train": count_jsonl_lines(OUTPUT_DIR / "train.jsonl"),
    "validation": count_jsonl_lines(OUTPUT_DIR / "validation.jsonl"),
    "test": count_jsonl_lines(OUTPUT_DIR / "test.jsonl")
}

print("Saved JSONL file sizes:")
print(saved_counts)

print("\nExpected original sizes:")
print({
    "train": train_example_count,
    "validation": validation_example_count,
    "test": test_example_count
})

if saved_counts["train"] != train_example_count:
    raise ValueError("Saved train size does not match the original train size.")

if saved_counts["validation"] != validation_example_count:
    raise ValueError("Saved validation size does not match the original validation size.")

if saved_counts["test"] != test_example_count:
    raise ValueError("Saved test size does not match the original test size.")

print("Success: saved files have the expected sizes.")

Saved JSONL file sizes:
{'train': 3990, 'validation': 570, 'test': 1140}

Expected original sizes:
{'train': 3990, 'validation': 570, 'test': 1140}
Success: saved files have the expected sizes.
